In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/default/raw_data/raw_cuts.csv',
  format => 'csv',
  header => true,
  multiLine => true
)
LIMIT 2;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_cuts AS
SELECT
  `Institution` AS institution,
  `State` AS state,
  `Cut Type` AS cut_type,
  `Program/Department` AS program_department,
  `Announcement Date` AS announcement_date,
  `Effective Term` AS effective_term,
  `Students Affected` AS students_affected,
  `Faculty/Staff Affected` AS faculty_staff_affected,
  `Primary Reason` AS primary_reason,
  `Status` AS status,
  `Source URL` AS source_url,
  `Notes` AS notes
FROM read_files(
  '/Volumes/workspace/default/raw_data/raw_cuts.csv',
  format => 'csv',
  header => true,
  inferSchema => true,
  multiLine => true
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_cuts AS
SELECT
  md5(concat_ws('|',
    trim(institution),
    trim(program_department),
    cast(try_cast(announcement_date AS DATE) AS STRING)
  )) AS cut_key,
  trim(institution) AS institution,
  trim(state) AS state,
  trim(cut_type) AS cut_type,
  trim(program_department) AS program_department,
  try_cast(announcement_date AS DATE) AS announcement_date,
  trim(effective_term) AS effective_term,
  try_cast(students_affected AS INT) AS students_affected,
  try_cast(faculty_staff_affected AS INT) AS faculty_staff_affected,
  trim(primary_reason) AS primary_reason,
  trim(status) AS status,
  trim(source_url) AS source_url,
  trim(notes) AS notes
FROM workspace.default.bronze_cuts;

In [0]:
%sql
SELECT
  count(*) AS silver_rows,
  count(DISTINCT cut_key) AS distinct_keys,
  sum(CASE WHEN source_url IS NULL THEN 1 ELSE 0 END) AS missing_source_url
FROM workspace.default.silver_cuts;

SELECT institution, students_affected, cut_key
FROM workspace.default.silver_cuts
WHERE institution LIKE 'Delta State%';

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_cuts_by_state AS
SELECT
  state,
  count(*) AS cuts,
  count(DISTINCT institution) AS institutions,
  sum(faculty_staff_affected) AS faculty_staff_affected
FROM workspace.default.silver_cuts
GROUP BY state
ORDER BY cuts DESC;

In [0]:
%sql
SELECT
  count(*) AS silver_rows,
  count(DISTINCT cut_key) AS distinct_keys,
  sum(CASE WHEN source_url IS NULL THEN 1 ELSE 0 END) AS missing_source_url
FROM workspace.default.silver_cuts;

In [0]:
%sql
SELECT * FROM workspace.default.gold_cuts_by_state
ORDER BY cuts DESC
LIMIT 5;